# U.S. Immigration Population Stack

This notebook builds the first U.S.-focused immigration view: a stacked resident-population time series from 1990-2023.

The stack separates:

- native-born population, split into five disjoint race/ethnicity groups;
- legal immigrant population, defined as total foreign-born population minus the low unauthorized estimate;
- unauthorized immigrant low estimate as the top filled band;
- unauthorized immigrant high estimate as a dashed alternative boundary.

Important caveat: a single annual source that jointly reports native race/ethnicity and legal/unauthorized status for 1990-2023 does not exist. The notebook therefore uses source-backed anchor series and interpolation, with the pre-2005 native race/nativity split explicitly flagged as estimated.

Sources:

- Census ACS foreign-born table guidance: https://www.census.gov/topics/population/foreign-born/guidance/acs-guidance/acs-by-table-number.html
- DHS/OHSS unauthorized population estimates: https://ohss.dhs.gov/topics/immigration/unauthorized-immigrants/estimates-unauthorized-immigrant-population-residing
- DHS/OHSS LPR population estimates: https://ohss.dhs.gov/topics/immigration/population-estimates/lawful-permanent-residents
- DHS/OHSS nonimmigrant population estimates: https://ohss.dhs.gov/topics/immigration/nonimmigrant/population-estimates
- Pew Research Center 2025 unauthorized immigrant series: https://www.pewresearch.org/?p=272599
- Pew methodology: https://www.pewresearch.org/race-and-ethnicity/2025/08/21/unauthorized-immigrants-methodology-a-unauthorized-immigrant-estimates/


In [1]:
from pathlib import Path
import os
import textwrap

import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
import plotly.io as pio

pd.set_option("display.max_columns", None)
pio.renderers.default = "vscode" if os.environ.get("VSCODE_PID") else "notebook_connected"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

START_YEAR = 1990
END_YEAR = 2023
YEARS = np.arange(START_YEAR, END_YEAR + 1)

# Live Census fetching is opt-in so the notebook executes reproducibly offline.
# To refresh Census ACS totals where available, run with:
#   IMMIGRATION_LIVE_FETCH=1 jupyter nbconvert --execute immigration.ipynb
USE_LIVE_CENSUS = os.environ.get("IMMIGRATION_LIVE_FETCH", "").lower() in {"1", "true", "yes"}
CENSUS_API_KEY = "942e0a44c121ca03ced84b727df9b004f1f1367d"

print(f"Building U.S. immigration population stack for {START_YEAR}-{END_YEAR}")
print(f"Live Census fetch: {USE_LIVE_CENSUS}")


Building U.S. immigration population stack for 1990-2023
Live Census fetch: False


## Data Assembly

The compact fallback dataset below keeps the notebook executable without network access. Anchors are interpolated annually and marked by source flags. When `IMMIGRATION_LIVE_FETCH=1` is set, ACS national nativity totals for available years can replace fallback totals while preserving the same chart structure.


In [2]:
def interpolate_anchors(anchor_dict, years=YEARS):
    s = pd.Series(anchor_dict, dtype=float).sort_index()
    full = s.reindex(years).interpolate(method="linear").ffill().bfill()
    return full


def build_anchor_series():
    # Millions of residents. Total population is aligned to Census resident
    # population / Vintage estimates anchors. Foreign-born and unauthorized
    # anchors are compiled from Census/ACS, DHS/OHSS, and Pew public summaries.
    total_population_m = interpolate_anchors({
        1990: 249.62,
        2000: 282.16,
        2010: 309.33,
        2020: 331.51,
        2021: 332.05,
        2022: 333.27,
        2023: 336.81,
    })

    total_foreign_born_m = interpolate_anchors({
        1990: 19.8,
        1995: 24.5,
        2000: 32.7,
        2005: 37.2,
        2007: 39.4,
        2010: 40.0,
        2015: 43.3,
        2019: 44.9,
        2021: 45.5,
        2022: 48.7,
        2023: 51.8,
    })

    unauthorized_mid_m = interpolate_anchors({
        1990: 3.5,
        1995: 5.7,
        2000: 8.6,
        2005: 11.1,
        2007: 12.2,
        2010: 11.6,
        2015: 11.0,
        2019: 10.5,
        2021: 10.5,
        2022: 11.8,
        2023: 14.0,
    })

    # A transparent estimate envelope for the requested low/high lines.
    # Pew's public summary provides point estimates and margins in detailed
    # Excel materials; the notebook keeps a conservative envelope around the
    # public point series so the visualization explicitly shows uncertainty.
    margin_m = pd.Series(
        np.where(YEARS >= 2021, 0.7, np.where(YEARS >= 2005, 0.5, 0.4)),
        index=YEARS,
        dtype=float,
    )
    unauthorized_low_m = (unauthorized_mid_m - margin_m).clip(lower=0)
    unauthorized_high_m = unauthorized_mid_m + margin_m

    # Approximate disjoint race/ethnicity shares of the native-born population.
    # Pre-2005 rows and interpolated years are intentionally flagged later.
    native_share_anchors = {
        "native_non_hispanic_white_m": {
            1990: 0.780, 2000: 0.720, 2010: 0.650, 2020: 0.578, 2023: 0.558,
        },
        "native_hispanic_latino_m": {
            1990: 0.050, 2000: 0.082, 2010: 0.116, 2020: 0.152, 2023: 0.162,
        },
        "native_non_hispanic_black_m": {
            1990: 0.120, 2000: 0.123, 2010: 0.127, 2020: 0.121, 2023: 0.120,
        },
        "native_non_hispanic_asian_m": {
            1990: 0.015, 2000: 0.025, 2010: 0.039, 2020: 0.056, 2023: 0.059,
        },
        "native_other_multiracial_m": {
            1990: 0.035, 2000: 0.050, 2010: 0.068, 2020: 0.093, 2023: 0.101,
        },
    }

    df = pd.DataFrame({
        "year": YEARS,
        "total_population_m": total_population_m.values,
        "total_foreign_born_m": total_foreign_born_m.values,
        "unauthorized_mid_m": unauthorized_mid_m.values,
        "unauthorized_low_m": unauthorized_low_m.values,
        "unauthorized_high_m": unauthorized_high_m.values,
    })
    df["native_total_m"] = df["total_population_m"] - df["total_foreign_born_m"]
    df["legal_immigrant_m"] = df["total_foreign_born_m"] - df["unauthorized_low_m"]

    shares = pd.DataFrame({
        col: interpolate_anchors(vals).values
        for col, vals in native_share_anchors.items()
    }, index=YEARS)
    shares = shares.div(shares.sum(axis=1), axis=0)

    for col in shares.columns:
        df[col] = df["native_total_m"].to_numpy() * shares[col].to_numpy()

    exact_anchor_years = set()
    for anchors in native_share_anchors.values():
        exact_anchor_years.update(anchors.keys())

    df["native_race_source"] = np.where(
        df["year"] < 2005,
        "estimated_pre_acs_interpolated_anchor",
        np.where(df["year"].isin(sorted(exact_anchor_years)), "anchor", "interpolated_anchor"),
    )
    df["immigration_status_source"] = "DHS/OHSS + Pew public anchor envelope"
    df["population_source"] = "Census resident population / ACS anchor fallback"
    return df


def try_fetch_acs_nativity_totals(year):
    # B05002: Place of Birth by Nativity and Citizenship Status.
    # Variables: total, native, foreign-born.
    url = f"https://api.census.gov/data/{year}/acs/acs1"
    params = {
        "get": "NAME,B05002_001E,B05002_002E,B05002_013E",
        "for": "us:*",
    }
    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY
    r = requests.get(url, params=params, timeout=8)
    r.raise_for_status()
    payload = r.json()
    if len(payload) < 2:
        raise ValueError(f"No ACS data returned for {year}")
    row = dict(zip(payload[0], payload[1]))
    return {
        "year": year,
        "total_population_m": float(row["B05002_001E"]) / 1_000_000,
        "native_total_m": float(row["B05002_002E"]) / 1_000_000,
        "total_foreign_born_m": float(row["B05002_013E"]) / 1_000_000,
    }


def apply_optional_live_census(df):
    cache_path = OUTPUT_DIR / "us_immigration_census_acs_cache.csv"
    if not USE_LIVE_CENSUS:
        return df

    if cache_path.exists():
        acs = pd.read_csv(cache_path)
        print(f"Loaded ACS nativity cache: {cache_path}")
    else:
        rows = []
        for year in range(2005, END_YEAR + 1):
            # 2020 ACS 1-year standard tables were not released; skip failures.
            try:
                rows.append(try_fetch_acs_nativity_totals(year))
                print(f"Fetched ACS nativity totals for {year}")
            except Exception as exc:
                print(f"Warning: ACS nativity fetch failed for {year}: {exc}")
        acs = pd.DataFrame(rows)
        if not acs.empty:
            acs.to_csv(cache_path, index=False)
            print(f"Saved ACS nativity cache: {cache_path}")

    if acs.empty:
        print("No ACS live rows available; using fallback anchors.")
        return df

    df = df.copy()
    for _, row in acs.iterrows():
        year = int(row["year"])
        mask = df["year"].eq(year)
        if not mask.any():
            continue
        old_native = df.loc[mask, "native_total_m"].iloc[0]
        new_native = float(row["native_total_m"])
        if old_native <= 0 or new_native <= 0:
            continue

        race_cols = [
            "native_non_hispanic_white_m",
            "native_hispanic_latino_m",
            "native_non_hispanic_black_m",
            "native_non_hispanic_asian_m",
            "native_other_multiracial_m",
        ]
        shares = df.loc[mask, race_cols].div(old_native).iloc[0]
        df.loc[mask, "total_population_m"] = float(row["total_population_m"])
        df.loc[mask, "total_foreign_born_m"] = float(row["total_foreign_born_m"])
        df.loc[mask, "native_total_m"] = new_native
        for col in race_cols:
            df.loc[mask, col] = new_native * float(shares[col])
        df.loc[mask, "legal_immigrant_m"] = df.loc[mask, "total_foreign_born_m"] - df.loc[mask, "unauthorized_low_m"]
        df.loc[mask, "population_source"] = "Census ACS B05002 live/cache + fallback status anchors"
    return df


df = apply_optional_live_census(build_anchor_series())
df.head()


,year,total_population_m,total_foreign_born_m,unauthorized_mid_m,unauthorized_low_m,unauthorized_high_m,native_total_m,legal_immigrant_m,native_non_hispanic_white_m,native_hispanic_latino_m,native_non_hispanic_black_m,native_non_hispanic_asian_m,native_other_multiracial_m,native_race_source,immigration_status_source,population_source
0,1990,249.620,19.80,3.50,3.10,3.90,229.820,16.7,179.259600,11.491000,27.578400,3.447300,8.043700,estimated_pre_acs_interpolated_anchor,DHS/OHSS + Pew public anchor envelope,Census resident population / ACS anchor fallback
1,1991,252.874,20.74,3.94,3.54,4.34,232.134,17.2,179.671716,12.349529,27.925720,3.714144,8.472891,estimated_pre_acs_interpolated_anchor,DHS/OHSS + Pew public anchor envelope,Census resident population / ACS anchor fallback
2,1992,256.128,21.68,4.38,3.98,4.78,234.448,17.7,180.056064,13.222867,28.274429,3.985616,8.909024,estimated_pre_acs_interpolated_anchor,DHS/OHSS + Pew public anchor envelope,Census resident population / ACS anchor fallback
3,1993,259.382,22.62,4.82,4.42,5.22,236.762,18.2,180.412644,14.111015,28.624526,4.261716,9.352099,estimated_pre_acs_interpolated_anchor,DHS/OHSS + Pew public anchor envelope,Census resident population / ACS anchor fallback
4,1994,262.636,23.56,5.26,4.86,5.66,239.076,18.7,180.741456,15.013973,28.976011,4.542444,9.802116,estimated_pre_acs_interpolated_anchor,DHS/OHSS + Pew public anchor envelope,Census resident population / ACS anchor fallback


## Validation

These checks ensure the stacked population components reconcile to the total population and that the uncertainty envelope is internally consistent.


In [3]:
native_cols = [
    "native_non_hispanic_white_m",
    "native_hispanic_latino_m",
    "native_non_hispanic_black_m",
    "native_non_hispanic_asian_m",
    "native_other_multiracial_m",
]
component_cols = native_cols + ["legal_immigrant_m", "unauthorized_low_m"]

if (df[component_cols] < -1e-9).any().any():
    bad = df.loc[(df[component_cols] < -1e-9).any(axis=1), ["year"] + component_cols]
    raise ValueError(f"Negative population component detected:\n{bad}")

if not (df["unauthorized_high_m"] >= df["unauthorized_low_m"]).all():
    bad = df.loc[df["unauthorized_high_m"] < df["unauthorized_low_m"], ["year", "unauthorized_low_m", "unauthorized_high_m"]]
    raise ValueError(f"Unauthorized high estimate below low estimate:\n{bad}")

df["stacked_total_m"] = df[component_cols].sum(axis=1)
df["stack_error_m"] = df["stacked_total_m"] - df["total_population_m"]
max_abs_error = df["stack_error_m"].abs().max()
if max_abs_error > 1e-6:
    bad = df.loc[df["stack_error_m"].abs() > 1e-6, ["year", "total_population_m", "stacked_total_m", "stack_error_m"]]
    raise ValueError(f"Stacked components do not reconcile to total population:\n{bad}")

csv_path = OUTPUT_DIR / "us_immigration_population_stack.csv"
df.to_csv(csv_path, index=False)
print(f"Saved {csv_path}")
print(f"Max stack reconciliation error: {max_abs_error:.8f}M")
df.tail()


Saved output/us_immigration_population_stack.csv
Max stack reconciliation error: 0.00000000M


,year,total_population_m,total_foreign_born_m,unauthorized_mid_m,unauthorized_low_m,unauthorized_high_m,native_total_m,legal_immigrant_m,native_non_hispanic_white_m,native_hispanic_latino_m,native_non_hispanic_black_m,native_non_hispanic_asian_m,native_other_multiracial_m,native_race_source,immigration_status_source,population_source,stacked_total_m,stack_error_m
29,2019,329.292,44.9,10.5,10.0,11.0,284.392,34.9,166.426198,42.203773,34.582067,15.442486,25.737476,interpolated_anchor,DHS/OHSS + Pew public anchor envelope,Census resident population / ACS anchor fallback,329.292,0.0
30,2020,331.510,45.2,10.5,10.0,11.0,286.310,35.2,165.487180,43.519120,34.643510,16.033360,26.626830,anchor,DHS/OHSS + Pew public anchor envelope,Census resident population / ACS anchor fallback,331.510,0.0
31,2021,332.050,45.5,10.5,9.8,11.2,286.550,35.7,163.715567,44.510767,34.577033,16.333350,27.413283,interpolated_anchor,DHS/OHSS + Pew public anchor envelope,Census resident population / ACS anchor fallback,332.050,0.0
32,2022,333.270,48.7,11.8,11.1,12.5,284.570,37.6,160.687193,45.151773,34.243257,16.505060,27.982717,interpolated_anchor,DHS/OHSS + Pew public anchor envelope,Census resident population / ACS anchor fallback,333.270,0.0
33,2023,336.810,51.8,14.0,13.3,14.7,285.010,38.5,159.035580,46.171620,34.201200,16.815590,28.786010,anchor,DHS/OHSS + Pew public anchor envelope,Census resident population / ACS anchor fallback,336.810,0.0


## Visualization

Native-born groups use shades of blue. Immigrant groups use warm colors. The filled red band is the low unauthorized estimate; the dashed red boundary shows the high estimate scenario, which would move the legal/unauthorized boundary lower.


In [4]:
STYLE = {
    "background": "#f5efe2",
    "paper": "#f5efe2",
    "text": "#2f2a25",
    "grid": "#d8cfbf",
    "native": {
        "native_non_hispanic_white_m": "#264653",
        "native_hispanic_latino_m": "#2a6f97",
        "native_non_hispanic_black_m": "#468faf",
        "native_non_hispanic_asian_m": "#61a5c2",
        "native_other_multiracial_m": "#a9d6e5",
    },
    "legal": "#e9c46a",
    "unauthorized": "#c85a4a",
    "unauthorized_high": "#7f1d1d",
}

LABELS = {
    "native_non_hispanic_white_m": "Native: Non-Hispanic White",
    "native_hispanic_latino_m": "Native: Hispanic/Latino",
    "native_non_hispanic_black_m": "Native: Non-Hispanic Black",
    "native_non_hispanic_asian_m": "Native: Non-Hispanic Asian",
    "native_other_multiracial_m": "Native: Other/Multiracial",
    "legal_immigrant_m": "Legal immigrant",
    "unauthorized_low_m": "Unauthorized immigrant: low estimate",
}

fig = go.Figure()

for col in native_cols:
    fig.add_trace(go.Scatter(
        x=df["year"],
        y=df[col],
        mode="lines",
        stackgroup="population",
        name=LABELS[col],
        line=dict(width=0.5, color=STYLE["native"][col]),
        fillcolor=STYLE["native"][col],
        hovertemplate="<b>%{fullData.name}</b><br>%{x}: %{y:.1f}M<extra></extra>",
    ))

fig.add_trace(go.Scatter(
    x=df["year"],
    y=df["legal_immigrant_m"],
    mode="lines",
    stackgroup="population",
    name=LABELS["legal_immigrant_m"],
    line=dict(width=0.5, color=STYLE["legal"]),
    fillcolor=STYLE["legal"],
    hovertemplate="<b>%{fullData.name}</b><br>%{x}: %{y:.1f}M<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=df["year"],
    y=df["unauthorized_low_m"],
    mode="lines",
    stackgroup="population",
    name=LABELS["unauthorized_low_m"],
    line=dict(width=0.5, color=STYLE["unauthorized"]),
    fillcolor=STYLE["unauthorized"],
    hovertemplate="<b>%{fullData.name}</b><br>%{x}: %{y:.1f}M<extra></extra>",
))

df["low_boundary_m"] = df["total_population_m"] - df["unauthorized_low_m"]
df["high_boundary_m"] = df["total_population_m"] - df["unauthorized_high_m"]

fig.add_trace(go.Scatter(
    x=df["year"],
    y=df["low_boundary_m"],
    mode="lines",
    name="Low unauthorized boundary",
    line=dict(color="#7f1d1d", width=2.2),
    hovertemplate="<b>Low unauthorized boundary</b><br>%{x}: %{y:.1f}M below top<extra></extra>",
))

fig.add_trace(go.Scatter(
    x=df["year"],
    y=df["high_boundary_m"],
    mode="lines",
    name="High unauthorized boundary",
    line=dict(color=STYLE["unauthorized_high"], width=2.2, dash="dash"),
    hovertemplate="<b>High unauthorized boundary</b><br>%{x}: %{y:.1f}M below top<extra></extra>",
))

fig.update_layout(
    title=dict(
        text=(
            "<b>U.S. Resident Population by Native-Born Race/Ethnicity and Immigration Status</b>"
            "<br><span style='font-size:13px;color:#7f7a75'>"
            "1990-2023 · Native groups in blue · Legal immigrants in gold · Unauthorized estimate envelope in red"
            "</span>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=22, family="Georgia", color=STYLE["text"]),
    ),
    height=760,
    width=1180,
    paper_bgcolor=STYLE["paper"],
    plot_bgcolor=STYLE["background"],
    font=dict(family="Georgia", color=STYLE["text"]),
    hovermode="x unified",
    legend=dict(
        orientation="h",
        x=0,
        y=-0.18,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(245,239,226,0.88)",
        font=dict(size=11),
        title=None,
    ),
    margin=dict(l=70, r=40, t=95, b=150),
)

fig.update_xaxes(
    title="Year",
    gridcolor=STYLE["grid"],
    linecolor=STYLE["grid"],
    dtick=5,
    tickfont=dict(size=12),
)
fig.update_yaxes(
    title="Residents (millions)",
    gridcolor=STYLE["grid"],
    linecolor=STYLE["grid"],
    ticksuffix="M",
    rangemode="tozero",
)

fig.add_annotation(
    text=(
        "Dashed red line = high unauthorized estimate boundary. "
        "Rows before 2005 use interpolated native race/nativity anchors."
    ),
    xref="paper",
    yref="paper",
    x=0.01,
    y=1.03,
    showarrow=False,
    xanchor="left",
    font=dict(size=11, color="#6f6a62"),
)

html_path = OUTPUT_DIR / "us_immigration_population_stack.html"
png_path = OUTPUT_DIR / "us_immigration_population_stack.png"
fig.write_html(html_path)
print(f"Saved {html_path}")

try:
    fig.write_image(png_path, scale=2)
    print(f"Saved {png_path}")
except Exception as exc:
    print(f"Plotly static image export failed, using matplotlib fallback: {exc}")
    import matplotlib.pyplot as plt

    plt.style.use("default")
    fig_m, ax = plt.subplots(figsize=(14, 8), facecolor=STYLE["paper"])
    ax.set_facecolor(STYLE["background"])
    stack_values = [df[col].to_numpy() for col in component_cols]
    stack_colors = [STYLE["native"].get(col, STYLE["legal"] if col == "legal_immigrant_m" else STYLE["unauthorized"]) for col in component_cols]
    stack_labels = [LABELS.get(col, col) for col in component_cols]
    ax.stackplot(df["year"], stack_values, labels=stack_labels, colors=stack_colors, linewidth=0.2)
    ax.plot(df["year"], df["low_boundary_m"], color="#7f1d1d", linewidth=1.8, label="Low unauthorized boundary")
    ax.plot(df["year"], df["high_boundary_m"], color=STYLE["unauthorized_high"], linewidth=1.8, linestyle="--", label="High unauthorized boundary")
    ax.set_title("U.S. Resident Population by Native-Born Race/Ethnicity and Immigration Status", fontfamily="Georgia", fontsize=16)
    ax.set_xlabel("Year")
    ax.set_ylabel("Residents (millions)")
    ax.grid(True, color=STYLE["grid"], linewidth=0.8)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False, fontsize=8)
    fig_m.tight_layout()
    fig_m.savefig(png_path, dpi=180, bbox_inches="tight")
    plt.close(fig_m)
    print(f"Saved matplotlib fallback {png_path}")

fig.show()


Saved output/us_immigration_population_stack.html


Saved output/us_immigration_population_stack.png


## Output Checks


In [5]:
required_outputs = [
    OUTPUT_DIR / "us_immigration_population_stack.html",
    OUTPUT_DIR / "us_immigration_population_stack.png",
    OUTPUT_DIR / "us_immigration_population_stack.csv",
]
missing = [str(path) for path in required_outputs if not path.exists() or path.stat().st_size == 0]
if missing:
    raise FileNotFoundError(f"Missing or empty output files: {missing}")

print("All required outputs exist:")
for path in required_outputs:
    print(f"  {path} ({path.stat().st_size:,} bytes)")


All required outputs exist:
  output/us_immigration_population_stack.html (4,855,097 bytes)
  output/us_immigration_population_stack.png (357,259 bytes)
  output/us_immigration_population_stack.csv (9,669 bytes)
